# Business Data Analyst Assistant Toolkit — V3
**Adaptive, schema-aware data analysis | Jupyter Notebook portfolio project**

V3 reads the column names that actually exist in an uploaded CSV or Excel file. It does not require or invent standard business field names. It profiles the dataset, detects data types, lets the analyst explicitly assign optional analytical roles, performs general-purpose quality checks, and exports a formatted Excel report.

**Design principle:** detect structure automatically; confirm business meaning explicitly.

## Quick start
1. Place one CSV, XLSX, XLSM, or XLS file in `data/raw/` (or set `INPUT_FILE` below).
2. Run the notebook from top to bottom.
3. Review the detected schema and data-quality profile.
4. Optionally assign date, amount, category, and status roles using the exact column names displayed by the notebook.
5. Review the generated Excel workbook in `reports/`.

The original source file is never modified. Synthetic sample data is generated only when no supported input file is found.

## 1. Environment and project folders

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)

PROJECT_DIR = Path.cwd() / "business_data_analyst_assistant_v3"
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
CHART_DIR = REPORT_DIR / "charts"
LOG_DIR = PROJECT_DIR / "logs"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR, CHART_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "processing_log.txt"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)
logging.info("V3 workflow initialized.")
print("Project directory:", PROJECT_DIR.resolve())
print("Input directory:", RAW_DIR.resolve())

## 2. Input settings

Set `INPUT_FILE` to a filename or full path if you want to select a particular file. Leave it as `None` to discover supported files in `data/raw/`.

If several files are present, the notebook lists them and selects the most recently modified file. Change `INPUT_FILE` to avoid ambiguity.

In [ ]:
SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xlsm", ".xls"}
INPUT_FILE = None  # Example: "monthly_export.xlsx"
CSV_ENCODING = "utf-8-sig"
EXCEL_SHEET = 0    # Sheet index or exact sheet name
HEADER_ROW = 0     # Zero-based header row

def discover_input_file():
    if INPUT_FILE:
        candidate = Path(INPUT_FILE).expanduser()
        if not candidate.is_absolute():
            candidate = RAW_DIR / candidate
        if not candidate.exists():
            raise FileNotFoundError(f"Configured input file was not found: {candidate}")
        if candidate.suffix.lower() not in SUPPORTED_EXTENSIONS:
            raise ValueError(f"Unsupported file type: {candidate.suffix}")
        return candidate

    candidates = [
        p for p in RAW_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
        and not p.name.startswith("~$")
    ]
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    print("Supported files discovered:")
    for item in candidates:
        print(f" - {item.name}")
    if len(candidates) > 1:
        print("Multiple files found; selecting the most recently modified file.")
    return candidates[0]

def load_dataset(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, encoding=CSV_ENCODING, header=HEADER_ROW)
    if suffix in {".xlsx", ".xlsm", ".xls"}:
        return pd.read_excel(path, sheet_name=EXCEL_SHEET, header=HEADER_ROW)
    raise ValueError(f"Unsupported extension: {suffix}")

source_path = discover_input_file()
if source_path is None:
    print("No input file found. Creating a clearly labelled synthetic demonstration dataset.")
    raw_df = pd.DataFrame({
        "Record Ref": ["A-100", "A-101", "A-102", "A-103", "A-104", "A-105"],
        "Logged On": ["2026-09-02", "2026-09-12", "2026-10-03", "2026-10-14", "2026-10-20", "2026-11-02"],
        "Channel Group": ["Online", "Retail", "Online", "Retail", "Partner", "Online"],
        "Net Figure": [1250.0, 900.0, 1450.0, 1100.0, 700.0, 1600.0],
        "Processing State": ["Complete", "Complete", "Pending", "Complete", "Complete", "Pending"]
    })
    SOURCE_NAME = "synthetic_demo_data"
    IS_SYNTHETIC = True
else:
    raw_df = load_dataset(source_path)
    SOURCE_NAME = source_path.name
    IS_SYNTHETIC = False

if raw_df.empty:
    raise ValueError("The selected dataset contains no data rows.")
if len(raw_df.columns) == 0:
    raise ValueError("No columns were detected. Check the header row setting.")

# Preserve original headers exactly; make duplicate headers visibly distinguishable.
original_headers = [str(c) for c in raw_df.columns]
seen = {}
unique_headers = []
for header in original_headers:
    seen[header] = seen.get(header, 0) + 1
    unique_headers.append(header if seen[header] == 1 else f"{header}__duplicate_{seen[header]}")
raw_df.columns = unique_headers

print(f"Source: {SOURCE_NAME}")
print(f"Rows: {len(raw_df):,} | Columns: {len(raw_df.columns):,}")
print("Detected columns:", list(raw_df.columns))
display(raw_df.head())

## 3. Schema discovery — actual columns only
This section reports the headers and inferred data types found in the loaded file. It does not rename columns or assign business meanings.

In [ ]:
def infer_column_kind(series):
    non_null = series.dropna()
    if non_null.empty:
        return "empty"
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    # Conservative date inference: classify only when most non-empty values parse.
    if non_null.dtype == "object" or pd.api.types.is_string_dtype(series):
        parsed = pd.to_datetime(non_null, errors="coerce")
        if parsed.notna().mean() >= 0.85 and len(non_null) >= 2:
            return "date-like (text; confirm)"
    if non_null.nunique() <= min(30, max(10, int(len(non_null) * 0.20))):
        return "categorical candidate"
    return "text / identifier candidate"

schema_rows = []
for col in raw_df.columns:
    s = raw_df[col]
    schema_rows.append({
        "column_name": col,
        "pandas_dtype": str(s.dtype),
        "detected_kind": infer_column_kind(s),
        "non_null_count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "unique_values": int(s.nunique(dropna=True)),
        "sample_values": " | ".join(map(str, s.dropna().head(3).tolist()))
    })
schema_df = pd.DataFrame(schema_rows)
display(schema_df)

## 4. Data quality profile

General-purpose checks are applied without assuming business-specific rules. Duplicate rows and missing values are reported, not silently deleted.

In [ ]:
quality_profile = pd.DataFrame({
    "column_name": raw_df.columns,
    "missing_count": [int(raw_df[c].isna().sum()) for c in raw_df.columns],
    "missing_percent": [round(float(raw_df[c].isna().mean() * 100), 2) for c in raw_df.columns],
    "distinct_count": [int(raw_df[c].nunique(dropna=True)) for c in raw_df.columns],
    "dtype": [str(raw_df[c].dtype) for c in raw_df.columns]
})
duplicate_mask = raw_df.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())
quality_summary = pd.DataFrame([
    {"metric": "row_count", "value": len(raw_df)},
    {"metric": "column_count", "value": len(raw_df.columns)},
    {"metric": "duplicate_row_count", "value": duplicate_count},
    {"metric": "fully_empty_rows", "value": int(raw_df.isna().all(axis=1).sum())},
    {"metric": "fully_empty_columns", "value": int(raw_df.isna().all(axis=0).sum())}
])
display(quality_summary)
display(quality_profile)
duplicate_rows = raw_df.loc[duplicate_mask].copy()
print("Duplicate rows are retained in the working dataset and separately reported.")

## 5. Optional analytical role selection

The notebook does not guess that a field is revenue, date, customer, or status. Use the exact detected column names to assign optional roles. Leave a role as `None` when it is not relevant.

The selected roles affect only the corresponding optional analysis.

In [ ]:
available_columns = list(raw_df.columns)

def choose_column(role_name, suggested_kind=None):
    print(f"\n{role_name} — available columns:")
    if suggested_kind:
        candidates = schema_df.loc[
            schema_df["detected_kind"].str.contains(suggested_kind, case=False, na=False),
            "column_name"
        ].tolist()
        print("Potential candidates (suggestions only):", candidates or "None")
    for i, name in enumerate(available_columns):
        print(f"  {i}: {name}")
    answer = input(f"Enter exact column name for {role_name}, or press Enter to skip: ").strip()
    if not answer:
        return None
    if answer not in available_columns:
        raise ValueError(f"{answer!r} is not an exact detected column name.")
    return answer

# Set these manually after reviewing the printed schema, or leave as None.
DATE_COLUMN = None
VALUE_COLUMN = None
CATEGORY_COLUMN = None
STATUS_COLUMN = None

# Optional interactive selection: change to True to be prompted.
PROMPT_FOR_ROLES = False

if PROMPT_FOR_ROLES:
    DATE_COLUMN = choose_column("date / time", "date")
    VALUE_COLUMN = choose_column("numeric measure", "numeric")
    CATEGORY_COLUMN = choose_column("category / grouping")
    STATUS_COLUMN = choose_column("status / state")

def validate_role(name, role):
    if role is not None and role not in available_columns:
        raise ValueError(f"{name} must exactly match a detected column name.")
for role_name, role_value in {
    "DATE_COLUMN": DATE_COLUMN,
    "VALUE_COLUMN": VALUE_COLUMN,
    "CATEGORY_COLUMN": CATEGORY_COLUMN,
    "STATUS_COLUMN": STATUS_COLUMN
}.items():
    validate_role(role_name, role_value)

role_mapping = {
    "date": DATE_COLUMN,
    "value": VALUE_COLUMN,
    "category": CATEGORY_COLUMN,
    "status": STATUS_COLUMN
}
display(pd.DataFrame([{"analytical_role": k, "selected_column": v} for k, v in role_mapping.items()]))

## 6. Working copy and conservative cleaning

The source DataFrame remains unchanged. This step creates a separate working copy, trims whitespace in string cell values, and attempts date conversion only for the explicitly selected date role.

In [ ]:
working_df = raw_df.copy(deep=True)
working_df.insert(0, "source_row_number", np.arange(2, len(working_df) + 2))

for col in raw_df.columns:
    if pd.api.types.is_object_dtype(working_df[col]) or pd.api.types.is_string_dtype(working_df[col]):
        working_df[col] = working_df[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )

if DATE_COLUMN:
    working_df[DATE_COLUMN] = pd.to_datetime(working_df[DATE_COLUMN], errors="coerce")

if VALUE_COLUMN:
    # Numeric conversion is explicit and reported; invalid values become missing, not zero.
    working_df[VALUE_COLUMN] = pd.to_numeric(working_df[VALUE_COLUMN], errors="coerce")

print("Working copy prepared.")
print("Source data unchanged:", raw_df.equals(raw_df.copy()))
display(working_df.head())

## 7. Validation and exception register

Validation is intentionally generic: missing values in selected analytical roles, failed date/number conversions, and duplicate rows. No records are automatically discarded.

In [ ]:
exceptions = []

def add_exception(mask, column, issue):
    for idx in working_df.index[mask.fillna(False)]:
        exceptions.append({
            "dataframe_index": idx,
            "source_row_number": working_df.at[idx, "source_row_number"],
            "column_name": column,
            "issue": issue,
            "observed_value": str(working_df.at[idx, column]) if column in working_df.columns else ""
        })

for role_name, col in role_mapping.items():
    if col:
        add_exception(working_df[col].isna(), col, f"Missing or unparseable selected {role_name} value")

for idx in working_df.index[duplicate_mask]:
    exceptions.append({
        "dataframe_index": idx,
        "source_row_number": working_df.at[idx, "source_row_number"],
        "column_name": "(entire row)",
        "issue": "Duplicate row",
        "observed_value": ""
    })

exception_df = pd.DataFrame(
    exceptions,
    columns=["dataframe_index", "source_row_number", "column_name", "issue", "observed_value"]
)
print(f"Exceptions recorded: {len(exception_df):,}")
display(exception_df.head(30))

## 8. General profiling and analysis
Numeric columns receive descriptive statistics. Categorical candidates receive frequency summaries. These are exploratory summaries, not claims about business meaning.

In [ ]:
numeric_columns = raw_df.select_dtypes(include="number").columns.tolist()
descriptive_stats = raw_df[numeric_columns].describe().T if numeric_columns else pd.DataFrame()

categorical_columns = [
    row["column_name"] for row in schema_rows
    if row["detected_kind"] == "categorical candidate"
]
frequency_tables = {}
for col in categorical_columns:
    frequency_tables[col] = (
        raw_df[col].astype("string").fillna("[Missing]")
        .value_counts(dropna=False).rename_axis("value").reset_index(name="count")
    )

print("Numeric columns:", numeric_columns)
display(descriptive_stats)
for col, table in frequency_tables.items():
    print(f"Frequency profile: {col}")
    display(table.head(30))

## 9. Optional selected-field summaries

When a value column is selected, the notebook calculates count, sum, mean, minimum, and maximum for its valid numeric values. When a category is also selected, it produces a grouped summary. A date role enables a monthly trend and month-over-month comparison.

In [ ]:
kpi_rows = []
grouped_summary = pd.DataFrame()
monthly_summary = pd.DataFrame()
mom_summary = pd.DataFrame()

if VALUE_COLUMN:
    values = working_df[VALUE_COLUMN].dropna()
    kpi_rows = [
        {"metric": "valid_value_count", "value": int(values.count())},
        {"metric": "sum", "value": float(values.sum())},
        {"metric": "mean", "value": float(values.mean()) if len(values) else np.nan},
        {"metric": "minimum", "value": float(values.min()) if len(values) else np.nan},
        {"metric": "maximum", "value": float(values.max()) if len(values) else np.nan}
    ]
    if CATEGORY_COLUMN:
        grouped_summary = (
            working_df.groupby(CATEGORY_COLUMN, dropna=False)[VALUE_COLUMN]
            .agg(record_count="count", total="sum", average="mean")
            .reset_index()
        )

    if DATE_COLUMN:
        trend_base = working_df.dropna(subset=[DATE_COLUMN]).copy()
        if not trend_base.empty:
            trend_base["_month"] = trend_base[DATE_COLUMN].dt.to_period("M").astype(str)
            monthly_summary = (
                trend_base.groupby("_month")[VALUE_COLUMN]
                .agg(record_count="count", total="sum", average="mean")
                .reset_index()
                .rename(columns={"_month": "month"})
            )
            monthly_summary["previous_month_total"] = monthly_summary["total"].shift(1)
            monthly_summary["change_amount"] = (
                monthly_summary["total"] - monthly_summary["previous_month_total"]
            )
            monthly_summary["change_percent"] = np.where(
                monthly_summary["previous_month_total"].ne(0),
                monthly_summary["change_amount"] / monthly_summary["previous_month_total"] * 100,
                np.nan
            )
            mom_summary = monthly_summary.copy()

kpi_df = pd.DataFrame(kpi_rows, columns=["metric", "value"])
print("Selected-field KPIs")
display(kpi_df)
if not grouped_summary.empty:
    print("Grouped summary")
    display(grouped_summary)
if not monthly_summary.empty:
    print("Monthly summary and month-over-month movement")
    display(monthly_summary)
if not VALUE_COLUMN:
    print("No value column selected; selected-field KPIs and MoM analysis are skipped.")

## 10. Visual summaries
Charts are generated only when the required fields have been explicitly selected and contain usable data.

In [ ]:
chart_paths = []
if VALUE_COLUMN and DATE_COLUMN and not monthly_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(monthly_summary["month"], monthly_summary["total"], marker="o")
    ax.set_title(f"Monthly total — {VALUE_COLUMN}")
    ax.set_xlabel("Month")
    ax.set_ylabel(VALUE_COLUMN)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    chart_path = CHART_DIR / "monthly_trend.png"
    fig.savefig(chart_path, dpi=150)
    chart_paths.append(chart_path)
    plt.show()
    plt.close(fig)

if VALUE_COLUMN and CATEGORY_COLUMN and not grouped_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    grouped_summary.plot(kind="bar", x=CATEGORY_COLUMN, y="total", legend=False, ax=ax)
    ax.set_title(f"Total {VALUE_COLUMN} by {CATEGORY_COLUMN}")
    ax.set_xlabel(CATEGORY_COLUMN)
    ax.set_ylabel(f"Total {VALUE_COLUMN}")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    chart_path = CHART_DIR / "category_summary.png"
    fig.savefig(chart_path, dpi=150)
    chart_paths.append(chart_path)
    plt.show()
    plt.close(fig)

if not chart_paths:
    print("No charts generated. Select suitable analytical roles to enable optional charts.")

## 11. Reconciliation checks

These checks compare exported row counts and selected-value totals against the in-memory working dataset. They are processing controls, not independent verification of the source system.

In [ ]:
reconciliation_rows = [
    {"check": "source_row_count", "expected": len(raw_df), "observed": len(working_df), "passed": len(raw_df) == len(working_df)}
]
if VALUE_COLUMN:
    source_total = pd.to_numeric(working_df[VALUE_COLUMN], errors="coerce").sum()
    reconciliation_rows.append({
        "check": f"selected_value_total__{VALUE_COLUMN}",
        "expected": float(source_total),
        "observed": float(working_df[VALUE_COLUMN].sum()),
        "passed": bool(np.isclose(source_total, working_df[VALUE_COLUMN].sum(), equal_nan=True))
    })
reconciliation_df = pd.DataFrame(reconciliation_rows)
display(reconciliation_df)

## 12. Excel report export

The workbook contains a run summary, detected schema, data-quality profile, cleaned working copy, exceptions, descriptive statistics, optional summaries, and reconciliation checks. Original column names are preserved in data tables.

In [ ]:
run_timestamp = datetime.now().isoformat(timespec="seconds")
run_summary = pd.DataFrame([
    {"item": "run_timestamp", "value": run_timestamp},
    {"item": "source_file", "value": SOURCE_NAME},
    {"item": "synthetic_demo_data", "value": IS_SYNTHETIC},
    {"item": "row_count", "value": len(raw_df)},
    {"item": "column_count", "value": len(raw_df.columns)},
    {"item": "duplicate_rows", "value": duplicate_count},
    {"item": "date_role", "value": DATE_COLUMN},
    {"item": "value_role", "value": VALUE_COLUMN},
    {"item": "category_role", "value": CATEGORY_COLUMN},
    {"item": "status_role", "value": STATUS_COLUMN}
])

report_path = REPORT_DIR / f"business_analysis_report_{datetime.now():%Y%m%d_%H%M%S}.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    run_summary.to_excel(writer, sheet_name="Run Summary", index=False)
    schema_df.to_excel(writer, sheet_name="Detected Schema", index=False)
    quality_summary.to_excel(writer, sheet_name="Quality Summary", index=False)
    quality_profile.to_excel(writer, sheet_name="Column Quality", index=False)
    working_df.to_excel(writer, sheet_name="Working Data", index=False)
    exception_df.to_excel(writer, sheet_name="Exceptions", index=False)
    descriptive_stats.to_excel(writer, sheet_name="Numeric Profile")
    kpi_df.to_excel(writer, sheet_name="Selected KPIs", index=False)
    grouped_summary.to_excel(writer, sheet_name="By Category", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly Trend", index=False)
    reconciliation_df.to_excel(writer, sheet_name="Reconciliation", index=False)

    from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
    from openpyxl.utils import get_column_letter

    workbook = writer.book
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E1F2")

    for ws in workbook.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = Border(bottom=thin_gray)
        for column_cells in ws.columns:
            values = [len(str(c.value)) if c.value is not None else 0 for c in list(column_cells)[:200]]
            width = min(max(max(values, default=10) + 2, 12), 48)
            ws.column_dimensions[get_column_letter(column_cells[0].column)].width = width
        ws.row_dimensions[1].height = 30

print("Excel report created:", report_path.resolve())

## 13. Run summary and responsible-use notes

Review exceptions and role assignments before interpreting results. A successful export does not prove that the source data is complete, accurate, or suitable for a business decision.

In [ ]:
final_summary = {
    "run_timestamp": run_timestamp,
    "source_file": SOURCE_NAME,
    "synthetic_demo_data": IS_SYNTHETIC,
    "rows": int(len(raw_df)),
    "columns": int(len(raw_df.columns)),
    "detected_column_names": list(map(str, raw_df.columns)),
    "duplicate_rows": duplicate_count,
    "exception_count": int(len(exception_df)),
    "selected_roles": role_mapping,
    "report_file": str(report_path.resolve())
}
print(json.dumps(final_summary, indent=2, default=str))
logging.info("V3 workflow completed. Source=%s Rows=%s Columns=%s Exceptions=%s",
             SOURCE_NAME, len(raw_df), len(raw_df.columns), len(exception_df))

## Operational notes and limitations

- Supported input formats: CSV, XLSX, XLSM, XLS.
- The notebook does not assume fixed business column names.
- Data type inference is heuristic. Date-like text and categorical candidates must be reviewed.
- Business roles are optional and must be explicitly assigned using exact detected column names.
- Numeric-looking identifiers should not automatically be treated as measures.
- Duplicate and missing records are flagged, not automatically removed.
- Month-over-month comparison requires an explicitly selected date column and numeric value column.
- Excel output is a working report, not an audited financial statement.
- Use synthetic or properly authorized, sanitized data when sharing this portfolio project.
- Excel `.xls` reading may require `xlrd`; install it only if that input format is needed.